Colab is making it easier than ever to integrate powerful Generative AI capabilities into your projects. We are launching public preview for a simple and intuitive Python library (google.colab.ai) to access state-of-the-art language models directly within Colab environments. All users have free access to most popular LLMs, while paid users have access to a wider selection of models. This means users can spend less time on configuration and set up and more time bringing their ideas to life. With just a few lines of code, you can now perform a variety of tasks:
- Generate text
- Translate languages
- Write creative content
- Categorize text

Happy Coding!


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/googlecolab/colabtools/blob/main/notebooks/Getting_started_with_google_colab_ai.ipynb)

In [1]:
# @title List available models
from google.colab import ai

ai.list_models()

['google/gemini-2.5-flash',
 'google/gemini-2.5-flash-lite',
 'google/gemini-3.5-flash']

Choosing a Model
The model names give you a hint about their capabilities and intended use:

Pro: These are the most capable models, ideal for complex reasoning, creative tasks, and detailed analysis.

Flash: These models are optimized for high speed and efficiency, making them great for summarization, chat applications, and tasks requiring rapid responses.

Gemma: These are lightweight, open-weight models suitable for a variety of text generation tasks and are great for experimentation.

In [ ]:
# @title Simple batch generation example
# Only text-to-text input/output is supported
from google.colab import ai

response = ai.generate_text("What is the capital of France?")
print(response)

The capital of France is **Paris**.



In [6]:
# @title Choose a different model
from google.colab import ai

response = ai.generate_text("What is the capital of England", model_name='google/gemini-3.5-flash')
print(response)

The capital of England is **London**. It is also the capital of the United Kingdom.


For longer text generations, you can stream the response. This displays the output token by token as it's generated, rather than waiting for the entire response to complete. This provides a more interactive and responsive experience. To enable this, simply set stream=True.

In [4]:
# @title Simple streaming example
from google.colab import ai

stream = ai.generate_text("Tell me a short story.", stream=True)
for text in stream:
  print(text, end='')

The shop of Elias Vance smelled of brass polish, old parchment, and the dry, metallic scent of tension springs. To the residents of Oakhaven, Elias was simply a master horologist—a quiet man with silver hair and hands that never shook, who could coax a dead grandfather clock back to life with a whisper and a drop of oil.

But Elias kept a secret in the left pocket of his waistcoat. It was a pocket watch of dull, unpolished iron, devoid of hands or a face. It had only a single brass crown at the top. 

Elias knew the rule of the watch, passed down through three generations of his family: *A minute stolen from the world is an hour stolen from the self.*

If he pressed the crown, the world would freeze. But for every frozen minute he spent walking among the statues of the paused world, his own body would age by an hour. Because of this, Elias used the watch with agonizing frugality. He had used it only a dozen times in his fifty years. Once, to catch a falling teacup holding his mother's 

In [5]:
#@title Text formatting setup
#code is not necessary for colab.ai, but is useful in fomatting text chunks
import sys

class LineWrapper:
    def __init__(self, max_length=80):
        self.max_length = max_length
        self.current_line_length = 0

    def print(self, text_chunk):
        i = 0
        n = len(text_chunk)
        while i < n:
            start_index = i
            while i < n and text_chunk[i] not in ' \n': # Find end of word
                i += 1
            current_word = text_chunk[start_index:i]

            delimiter = ""
            if i < n: # If not end of chunk, we found a delimiter
                delimiter = text_chunk[i]
                i += 1 # Consume delimiter

            if current_word:
                needs_leading_space = (self.current_line_length > 0)

                # Case 1: Word itself is too long for a line (must be broken)
                if len(current_word) > self.max_length:
                    if needs_leading_space: # Newline if current line has content
                        sys.stdout.write('\n')
                        self.current_line_length = 0
                    for char_val in current_word: # Break the long word
                        if self.current_line_length >= self.max_length:
                            sys.stdout.write('\n')
                            self.current_line_length = 0
                        sys.stdout.write(char_val)
                        self.current_line_length += 1
                # Case 2: Word doesn't fit on current line (print on new line)
                elif self.current_line_length + (1 if needs_leading_space else 0) + len(current_word) > self.max_length:
                    sys.stdout.write('\n')
                    sys.stdout.write(current_word)
                    self.current_line_length = len(current_word)
                # Case 3: Word fits on current line
                else:
                    if needs_leading_space:
                        # Define punctuation that should not have a leading space
                        # when they form an entire "word" (token) following another word.
                        no_leading_space_punctuation = {
                            ",", ".", ";", ":", "!", "?",        # Standard sentence punctuation
                            ")", "]", "}",                     # Closing brackets
                            "'s", "'S", "'re", "'RE", "'ve", "'VE", # Common contractions
                            "'m", "'M", "'ll", "'LL", "'d", "'D",
                            "n't", "N'T",
                            "...", "…"                          # Ellipses
                        }
                        if current_word not in no_leading_space_punctuation:
                            sys.stdout.write(' ')
                            self.current_line_length += 1
                    sys.stdout.write(current_word)
                    self.current_line_length += len(current_word)

            if delimiter == '\n':
                sys.stdout.write('\n')
                self.current_line_length = 0
            elif delimiter == ' ':
                # If line is full and a space delimiter arrives, it implies a wrap.
                if self.current_line_length >= self.max_length:
                    sys.stdout.write('\n')
                    self.current_line_length = 0

        sys.stdout.flush()


In [7]:
# @title Formatted streaming example
from google.colab import ai

wrapper = LineWrapper()
for chunk in ai.generate_text('Give me a long winded description about the evolution of the Roman Empire.', model_name='google/gemini-2.5-flash', stream=True):
  wrapper.print(chunk)

Ah, the Roman Empire – not a static monolith, but a colossal, ever-shifting
entity, a living, breathing testament to human ambition, ingenuity, and
ultimately, its own inherent contradictions. To trace its evolution is to embark
on an epic journey through a millennium of relentless expansion, profound
political transformation, unparalleled cultural efflorescence, and a slow,
agonizing decline that nevertheless gave birth to new worlds.

Our story does not begin with emperors, but with a collection of rugged,
tenacious Latins huddled by the Tiber, their nascent settlement of Rome steeped
in myth and hardened by an unforgiving Italian landscape. From these humble,
even mythical, origins – the twin founders Romulus and Remus, nurtured by a wolf
– emerged a monarchy, influenced by the more sophisticated Etruscans to the
north. This early period, shrouded in the mists of legend, laid the groundwork
for Roman identity: a fierce pride, a deep reverence for ancestral customs (the
*mos maiorum*